# Task 2 assumption audit notebook

Run this notebook on your final-task data and send me the printed **FINAL SUMMARY** plus the generated plots.

Goal: verify which architectural assumptions are actually supported by the data before committing to a model. It checks:

1. whether electrode order preserves physical adjacency,
2. whether physical neighbors have correlated responses,
3. whether time bins are strongly structured and correlated,
4. how strong the early response is,
5. how much pattern/frequency condition explains over the global mean,
6. whether frequency effects are smooth enough to justify continuous frequency features,
7. whether trial history/state dependence matters,
8. whether response matrices look low-rank enough for a latent decoder.

This notebook does **not** train the final model. It is an evidence-gathering notebook.

## 1. Configuration

Change only this cell if needed. It follows the same variables as the course notebooks.

In [ ]:
# -----------------------------
# Data selection
# -----------------------------
Network = 5
DIV = 40
group_data = False
test_data = False  # supervised training/audit file; _test has no responses

# -----------------------------
# Analysis options
# -----------------------------
RANDOM_SEED = 42
VAL_FRAC = 0.20
N_RANDOM_PAIRS = 30000      # pair samples for spatial-correlation diagnostics
N_RANDOM_TRIALS = 4000      # cap for expensive diagnostics; set None to use all
N_DISTANCE_BINS = 8
EARLY_BINS = [1, 2, 4, 8, 12, 20]
FREQ_BINS_FOR_BASELINE = 8
OUTPUT_DIR = "task2_assumption_audit_outputs"

# Optional: only set True if you want slower sanity checks later.
RUN_OPTIONAL_MODEL_SANITY_CHECKS = False


## 2. Imports and data loading

In [ ]:
import os, json, math, warnings, itertools, textwrap
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import pairwise_distances

try:
    from scipy.stats import spearmanr, pearsonr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    print("scipy not available; a few correlation p-values will be skipped")

try:
    from utils.data import load_data, save_data
    from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode
except Exception as e:
    raise ImportError("Could not import course utils. Run this notebook from the final-task repo root.") from e

os.makedirs(OUTPUT_DIR, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)

data = load_data(Network, DIV, group_data, test_mode=test_data)
stimulation_parameters, stimulation_patterns, binned_spike_train_responses, stimulation_times, impedance_map, electrodes = data

if test_data:
    data_test = load_data(Network, DIV, group_data, test_mode=True)
    stimulation_parameters_test, stimulation_patterns_test, binned_spike_train_responses_test, stimulation_times_test, _, _ = data_test
else:
    stimulation_parameters_test = stimulation_patterns_test = binned_spike_train_responses_test = stimulation_times_test = None

X = np.asarray(binned_spike_train_responses)
if X.ndim == 4 and X.shape[1] == 1:
    X = X[:, 0, :, :]
if X.ndim != 3:
    raise ValueError(f"Expected X shape (trials, electrodes, time) or (trials,1,electrodes,time), got {X.shape}")
X = X.astype(np.float32)

Z = np.asarray(stimulation_parameters)
patterns = Z[:, 1].astype(int) if Z.ndim == 2 and Z.shape[1] >= 2 else np.asarray(stimulation_patterns).reshape(-1).astype(int)
freqs = Z[:, 0].astype(float) if Z.ndim == 2 and Z.shape[1] >= 1 else np.full(X.shape[0], np.nan)

electrodes = np.asarray(electrodes)
T = X.shape[2]
N = X.shape[1]
M = X.shape[0]

print("Loaded data")
print("X shape (trials, electrodes, time):", X.shape)
print("stimulation_parameters shape:", Z.shape)
print("patterns shape:", patterns.shape, "unique:", np.unique(patterns))
print("freqs shape:", freqs.shape, "unique count:", len(np.unique(freqs)), "range:", (np.nanmin(freqs), np.nanmax(freqs)))
print("electrodes shape:", electrodes.shape)
print("impedance_map type/shape:", type(impedance_map), getattr(impedance_map, 'shape', None))


## 3. Helper functions

In [ ]:
def clip_prob(p, eps=1e-5):
    return np.clip(p, eps, 1 - eps)

def bce_from_prob(y, p):
    p = clip_prob(p)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

def logit(p, eps=1e-5):
    p = clip_prob(p, eps)
    return np.log(p / (1 - p))

def safe_corr(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    if len(a) < 3 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def try_extract_electrode_xy(impedance_map, electrodes):
    """
    Tries several common representations:
    1. electrodes is already (N,2) or (N,>=3) with coordinates
    2. impedance_map is a 2D array whose values encode electrode ids
    3. impedance_map is a dict-like object mapping electrode id -> coordinates

    Returns coords as shape (N,2), or None if extraction fails.
    """
    el = np.asarray(electrodes)
    # Case A: electrodes already contains coordinates.
    if el.ndim == 2 and el.shape[0] > 0 and el.shape[1] >= 2:
        # Heuristic: if first two columns have many unique values, they may be x/y.
        xy = el[:, :2].astype(float)
        if np.unique(xy[:, 0]).size > 2 and np.unique(xy[:, 1]).size > 2:
            return xy
        if el.shape[1] >= 3:
            xy = el[:, 1:3].astype(float)
            if np.unique(xy[:, 0]).size > 2 and np.unique(xy[:, 1]).size > 2:
                return xy

    # Use electrode IDs if electrodes is a vector or first column of matrix.
    if el.ndim == 1:
        ids = el
    elif el.ndim == 2:
        ids = el[:, 0]
    else:
        ids = np.arange(len(el))

    # Case B: impedance_map is dict-like.
    if isinstance(impedance_map, dict):
        coords = []
        ok = True
        for e in ids:
            key_options = [e, int(e) if np.issubdtype(type(e), np.number) else e, str(e)]
            found = None
            for k in key_options:
                if k in impedance_map:
                    found = impedance_map[k]
                    break
            if found is None:
                ok = False
                break
            arr = np.asarray(found).ravel()
            if arr.size < 2:
                ok = False
                break
            coords.append(arr[:2].astype(float))
        if ok:
            return np.asarray(coords)

    # Case C: impedance_map is a 2D grid containing electrode IDs.
    arr = np.asarray(impedance_map)
    if arr.ndim >= 2:
        # If more than 2D, squeeze singleton dims and try first plane.
        arr2 = np.squeeze(arr)
        if arr2.ndim > 2:
            arr2 = arr2[..., 0]
        if arr2.ndim == 2:
            coords = []
            ok = True
            for e in ids:
                loc = np.argwhere(arr2 == e)
                if loc.size == 0:
                    # try integer conversion
                    try:
                        loc = np.argwhere(arr2 == int(e))
                    except Exception:
                        pass
                if loc.size == 0:
                    ok = False
                    break
                y, x = loc[0]
                coords.append([float(x), float(y)])
            if ok:
                return np.asarray(coords)

    # Fallback: extract scatter offsets from the provided plotting helper.
    try:
        fig, ax = plt.subplots(figsize=(4, 4))
        map_colour_to_electrode(ax, impedance_map, electrodes)
        coords = None
        for collection in ax.collections:
            try:
                offsets = np.asarray(collection.get_offsets())
                if offsets.shape[0] == len(electrodes) and offsets.shape[1] >= 2:
                    coords = offsets[:, :2].astype(float)
                    break
            except Exception:
                pass
        plt.close(fig)
        if coords is not None:
            return coords
    except Exception:
        pass

    return None

def condition_keys(patterns, freqs, freq_round_decimals=8):
    fr = np.round(freqs.astype(float), freq_round_decimals)
    return np.array([f"p{int(p)}_f{float(f):.8g}" for p, f in zip(patterns, fr)])

def mean_lookup_predict(train_X, train_patterns, train_freqs, val_patterns, val_freqs,
                        mode="global", n_freq_bins=8, alpha=2.0):
    """
    Returns probability predictions for val_X shape as (n_val, N, T).
    Modes: global, pattern, pattern_freq_exact, pattern_freq_bin.
    alpha controls shrinkage toward global mean for sparse groups.
    """
    global_p = train_X.mean(axis=0)
    pred = np.empty((len(val_patterns), train_X.shape[1], train_X.shape[2]), dtype=np.float32)

    if mode == "global":
        pred[:] = global_p
        return pred

    groups = {}
    if mode == "pattern":
        train_group = train_patterns.astype(int)
        val_group = val_patterns.astype(int)
    elif mode == "pattern_freq_exact":
        train_group = condition_keys(train_patterns, train_freqs)
        val_group = condition_keys(val_patterns, val_freqs)
    elif mode == "pattern_freq_bin":
        # bins based on train frequencies only
        unique_f = np.unique(train_freqs)
        if len(unique_f) <= n_freq_bins:
            edges = np.r_[-np.inf, (unique_f[:-1] + unique_f[1:]) / 2, np.inf]
        else:
            edges = np.quantile(train_freqs, np.linspace(0, 1, n_freq_bins + 1))
            edges[0], edges[-1] = -np.inf, np.inf
            edges = np.unique(edges)
        train_bins = np.digitize(train_freqs, edges[1:-1])
        val_bins = np.digitize(val_freqs, edges[1:-1])
        train_group = np.array([f"p{int(p)}_b{int(b)}" for p, b in zip(train_patterns, train_bins)])
        val_group = np.array([f"p{int(p)}_b{int(b)}" for p, b in zip(val_patterns, val_bins)])
    else:
        raise ValueError(mode)

    unique_groups = np.unique(train_group)
    for g in unique_groups:
        idx = train_group == g
        n = int(idx.sum())
        # empirical Bayes shrinkage to avoid overconfident sparse condition means
        groups[g] = (train_X[idx].sum(axis=0) + alpha * global_p) / (n + alpha)

    for i, g in enumerate(val_group):
        pred[i] = groups.get(g, global_p)
    return pred

def savefig(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("saved", path)


## 4. Basic data audit

In [ ]:
summary = {}
summary["shape"] = {"trials": int(M), "electrodes": int(N), "time_bins": int(T)}
summary["n_patterns"] = int(len(np.unique(patterns)))
summary["n_frequencies"] = int(len(np.unique(freqs)))
summary["frequency_range"] = [float(np.nanmin(freqs)), float(np.nanmax(freqs))]
summary["global_spike_probability"] = float(X.mean())
summary["global_spikes_per_trial_mean"] = float(X.sum(axis=(1,2)).mean())
summary["global_spikes_per_trial_std"] = float(X.sum(axis=(1,2)).std())
summary["fraction_empty_trials"] = float((X.sum(axis=(1,2)) == 0).mean())

print(json.dumps(summary, indent=2))

# Condition counts
counts = pd.DataFrame({"pattern": patterns, "frequency": freqs}).groupby(["pattern", "frequency"]).size().reset_index(name="n")
print("\nCondition-count summary:")
print(counts["n"].describe())
print("\nSmallest condition counts:")
display(counts.sort_values("n").head(20))

plt.figure(figsize=(8, 4))
plt.hist(counts["n"], bins=30)
plt.xlabel("trials per exact pattern-frequency condition")
plt.ylabel("number of conditions")
plt.title("Condition repeat-count distribution")
savefig("condition_counts_hist.png")
plt.show()


## 5. Temporal structure and early response

This tests whether a time-aware decoder is justified, and whether the model should have an explicit early-response bias.

In [ ]:
psth = X.mean(axis=(0,1))
pop_count_by_trial_time = X.sum(axis=1)  # trials x time
trial_total = X.sum(axis=(1,2))

early_mass = {}
for k in EARLY_BINS:
    kk = min(k, T)
    early_mass[f"first_{kk}_bins_fraction_of_spikes"] = float(X[:, :, :kk].sum() / max(X.sum(), 1))
    early_mass[f"first_{kk}_bins_fraction_of_BCE_positions"] = float(kk / T)
summary["early_mass"] = early_mass
summary["psth_peak_bin"] = int(np.argmax(psth))
summary["psth_peak_prob"] = float(np.max(psth))
summary["psth_late_mean_after_20"] = float(psth[min(20,T-1):].mean())

plt.figure(figsize=(9, 4))
plt.plot(np.arange(T), psth, marker='o', markersize=3)
plt.axvline(np.argmax(psth), linestyle='--', label=f"peak bin {np.argmax(psth)}")
for k in [4, 8, 12, 20]:
    if k < T:
        plt.axvline(k, alpha=0.25)
plt.xlabel("time bin")
plt.ylabel("mean spike probability over trials/electrodes")
plt.title("Population PSTH / mean response over time")
plt.legend()
savefig("temporal_psth.png")
plt.show()

# Adjacent time-bin correlation across trial x electrode samples.
adj_corrs = []
flat = X.transpose(0,2,1).reshape(M*T, N)  # not used, keep memory reasonable below
for t in range(T-1):
    adj_corrs.append(safe_corr(X[:,:,t].ravel(), X[:,:,t+1].ravel()))
summary["temporal_adjacent_corr_mean"] = float(np.nanmean(adj_corrs))
summary["temporal_adjacent_corr_max"] = float(np.nanmax(adj_corrs))

plt.figure(figsize=(9, 3))
plt.plot(adj_corrs)
plt.xlabel("time bin t")
plt.ylabel("corr(x[t], x[t+1])")
plt.title("Adjacent-bin spike correlation across trials/electrodes")
savefig("temporal_adjacent_corr.png")
plt.show()

print("Early mass:")
print(json.dumps(early_mass, indent=2))
print("Mean adjacent-bin correlation:", summary["temporal_adjacent_corr_mean"])


## 6. Electrode-coordinate extraction and electrode-order adjacency

This is the key test for whether a CNN over electrode index is biologically meaningful. If consecutive electrode indices are not physically close, index-axis Conv2D is a bad inductive bias.

In [ ]:
coords = try_extract_electrode_xy(impedance_map, electrodes)
summary["coordinates_extracted"] = coords is not None

if coords is None:
    print("Could not extract physical coordinates. Spatial graph/CNN assumptions cannot be verified from current objects.")
else:
    coords = np.asarray(coords, dtype=float)
    print("coords shape:", coords.shape)
    summary["coord_x_range"] = [float(np.min(coords[:,0])), float(np.max(coords[:,0]))]
    summary["coord_y_range"] = [float(np.min(coords[:,1])), float(np.max(coords[:,1]))]

    plt.figure(figsize=(6, 5))
    sc = plt.scatter(coords[:,0], coords[:,1], c=np.arange(N), s=30)
    plt.colorbar(sc, label="electrode index in X")
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title("Electrode positions colored by response-tensor index")
    plt.xlabel("x")
    plt.ylabel("y")
    savefig("electrode_positions_colored_by_index.png")
    plt.show()

    D = pairwise_distances(coords)
    consecutive_dist = np.diag(D, k=1)
    rand_means = []
    for _ in range(1000):
        perm = rng.permutation(N)
        rand_means.append(np.mean(D[perm[:-1], perm[1:]]))
    rand_means = np.array(rand_means)

    observed_mean_consecutive_dist = float(np.mean(consecutive_dist))
    random_mean_consecutive_dist = float(np.mean(rand_means))
    percentile = float((rand_means <= observed_mean_consecutive_dist).mean())
    summary["electrode_order"] = {
        "mean_consecutive_physical_distance": observed_mean_consecutive_dist,
        "random_order_mean_distance": random_mean_consecutive_dist,
        "observed_percentile_vs_random_lower_is_more_spatial": percentile,
        "ratio_observed_to_random": observed_mean_consecutive_dist / random_mean_consecutive_dist,
    }

    # Correlation between index distance and physical distance.
    idx = np.arange(N)
    Ii, Jj = np.triu_indices(N, k=1)
    index_gap = np.abs(Ii - Jj)
    phys_dist = D[Ii, Jj]
    if SCIPY_AVAILABLE:
        rho, pval = spearmanr(index_gap, phys_dist)
    else:
        rho, pval = safe_corr(index_gap, phys_dist), np.nan
    summary["electrode_order"]["spearman_index_gap_vs_physical_distance"] = float(rho)
    summary["electrode_order"]["spearman_p_value"] = float(pval) if np.isfinite(pval) else None

    # Nearest-neighbor index gap.
    D_no_self = D.copy()
    np.fill_diagonal(D_no_self, np.inf)
    nn = np.argmin(D_no_self, axis=1)
    nn_index_gap = np.abs(np.arange(N) - nn)
    summary["electrode_order"]["median_index_gap_to_physical_nearest_neighbor"] = float(np.median(nn_index_gap))
    summary["electrode_order"]["fraction_physical_NN_with_index_gap_le_3"] = float((nn_index_gap <= 3).mean())

    print(json.dumps(summary["electrode_order"], indent=2))

    plt.figure(figsize=(7, 4))
    plt.hist(rand_means, bins=40, alpha=0.8, label="random order")
    plt.axvline(observed_mean_consecutive_dist, color='k', linestyle='--', label="observed order")
    plt.xlabel("mean distance between consecutive indices")
    plt.ylabel("count")
    plt.title("Is electrode tensor order spatially local?")
    plt.legend()
    savefig("electrode_order_vs_random.png")
    plt.show()

    plt.figure(figsize=(7,4))
    plt.hist(nn_index_gap, bins=30)
    plt.xlabel("index gap to physically nearest electrode")
    plt.ylabel("electrode count")
    plt.title("Do physical nearest neighbors have nearby tensor indices?")
    savefig("physical_nn_index_gap.png")
    plt.show()


## 7. Spatial response correlation vs physical distance

This tests whether nearby electrodes genuinely behave more similarly. If yes, graph/coordinate models are justified. If not, spatial inductive bias should be weak.

In [ ]:
if coords is None:
    print("Skipping spatial correlation because coordinates were not extracted.")
else:
    # Response feature per electrode: trial-level early spike counts + full spike counts.
    early_cut = min(12, T)
    F_full = X.sum(axis=2)              # trials x electrodes
    F_early = X[:, :, :early_cut].sum(axis=2)

    # cap trials for speed
    if N_RANDOM_TRIALS is not None and M > N_RANDOM_TRIALS:
        trial_idx = rng.choice(M, size=N_RANDOM_TRIALS, replace=False)
        F_full_use = F_full[trial_idx]
        F_early_use = F_early[trial_idx]
    else:
        F_full_use = F_full
        F_early_use = F_early

    D = pairwise_distances(coords)
    all_pairs = np.array(np.triu_indices(N, k=1)).T
    if len(all_pairs) > N_RANDOM_PAIRS:
        pairs = all_pairs[rng.choice(len(all_pairs), size=N_RANDOM_PAIRS, replace=False)]
    else:
        pairs = all_pairs

    dists = D[pairs[:,0], pairs[:,1]]
    corrs_full = np.array([safe_corr(F_full_use[:,i], F_full_use[:,j]) for i,j in pairs])
    corrs_early = np.array([safe_corr(F_early_use[:,i], F_early_use[:,j]) for i,j in pairs])

    valid = np.isfinite(corrs_full) & np.isfinite(corrs_early) & np.isfinite(dists)
    dists = dists[valid]
    corrs_full = corrs_full[valid]
    corrs_early = corrs_early[valid]

    # Bin by distance.
    edges = np.quantile(dists, np.linspace(0, 1, N_DISTANCE_BINS + 1))
    edges = np.unique(edges)
    bin_rows = []
    for b in range(len(edges)-1):
        mask = (dists >= edges[b]) & (dists <= edges[b+1] if b == len(edges)-2 else dists < edges[b+1])
        if mask.sum() == 0:
            continue
        bin_rows.append({
            "bin": b,
            "distance_low": float(edges[b]),
            "distance_high": float(edges[b+1]),
            "n_pairs": int(mask.sum()),
            "corr_full_mean": float(np.nanmean(corrs_full[mask])),
            "corr_early_mean": float(np.nanmean(corrs_early[mask])),
        })
    spatial_corr_df = pd.DataFrame(bin_rows)
    display(spatial_corr_df)

    if SCIPY_AVAILABLE:
        rho_full, p_full = spearmanr(dists, corrs_full)
        rho_early, p_early = spearmanr(dists, corrs_early)
    else:
        rho_full, p_full = safe_corr(dists, corrs_full), np.nan
        rho_early, p_early = safe_corr(dists, corrs_early), np.nan

    summary["spatial_correlation"] = {
        "spearman_distance_vs_full_count_corr": float(rho_full),
        "spearman_distance_vs_early_count_corr": float(rho_early),
        "p_full": float(p_full) if np.isfinite(p_full) else None,
        "p_early": float(p_early) if np.isfinite(p_early) else None,
        "nearest_distance_bin_full_corr_mean": float(spatial_corr_df.iloc[0]["corr_full_mean"]),
        "farthest_distance_bin_full_corr_mean": float(spatial_corr_df.iloc[-1]["corr_full_mean"]),
        "nearest_minus_farthest_full_corr": float(spatial_corr_df.iloc[0]["corr_full_mean"] - spatial_corr_df.iloc[-1]["corr_full_mean"]),
    }

    plt.figure(figsize=(7,4))
    centers = 0.5*(spatial_corr_df["distance_low"] + spatial_corr_df["distance_high"])
    plt.plot(centers, spatial_corr_df["corr_full_mean"], marker='o', label="full 80 bins")
    plt.plot(centers, spatial_corr_df["corr_early_mean"], marker='o', label=f"first {early_cut} bins")
    plt.xlabel("physical distance bin center")
    plt.ylabel("mean pairwise response correlation")
    plt.title("Do nearby electrodes respond similarly?")
    plt.legend()
    savefig("spatial_corr_vs_distance.png")
    plt.show()

    print(json.dumps(summary["spatial_correlation"], indent=2))


## 8. Baseline BCE: global mean vs condition means

This estimates how much Task 2 information is contained in pattern/frequency before using a neural architecture.

In [ ]:
# Random trial split.
idx = np.arange(M)
train_idx, val_idx = train_test_split(idx, test_size=VAL_FRAC, random_state=RANDOM_SEED, stratify=patterns if len(np.unique(patterns)) > 1 else None)

baseline_rows = []
for mode in ["global", "pattern", "pattern_freq_exact", "pattern_freq_bin"]:
    pred = mean_lookup_predict(X[train_idx], patterns[train_idx], freqs[train_idx], patterns[val_idx], freqs[val_idx],
                               mode=mode, n_freq_bins=FREQ_BINS_FOR_BASELINE, alpha=2.0)
    baseline_rows.append({
        "split": "random_trial",
        "baseline": mode,
        "bce": bce_from_prob(X[val_idx], pred),
    })

# Leave-frequency-out split if possible.
unique_freqs = np.unique(freqs)
if len(unique_freqs) >= 4:
    gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=RANDOM_SEED)
    tr, va = next(gss.split(idx, groups=freqs))
    train_f_idx, val_f_idx = idx[tr], idx[va]
    heldout_freqs = np.unique(freqs[val_f_idx])
    print("Held-out frequencies:", heldout_freqs)
    for mode in ["global", "pattern", "pattern_freq_exact", "pattern_freq_bin"]:
        pred = mean_lookup_predict(X[train_f_idx], patterns[train_f_idx], freqs[train_f_idx], patterns[val_f_idx], freqs[val_f_idx],
                                   mode=mode, n_freq_bins=FREQ_BINS_FOR_BASELINE, alpha=2.0)
        baseline_rows.append({
            "split": "leave_frequency_out",
            "baseline": mode,
            "bce": bce_from_prob(X[val_f_idx], pred),
        })
else:
    heldout_freqs = []

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

# Improvements over global.
for split in baseline_df["split"].unique():
    base = float(baseline_df[(baseline_df.split == split) & (baseline_df.baseline == "global")]["bce"].iloc[0])
    baseline_df.loc[baseline_df.split == split, "delta_vs_global"] = base - baseline_df.loc[baseline_df.split == split, "bce"]
    baseline_df.loc[baseline_df.split == split, "relative_improvement_vs_global"] = (base - baseline_df.loc[baseline_df.split == split, "bce"]) / base

display(baseline_df)
summary["baseline_bce"] = baseline_df.to_dict(orient="records")

plt.figure(figsize=(9,4))
for split in baseline_df["split"].unique():
    sub = baseline_df[baseline_df["split"] == split]
    plt.plot(sub["baseline"], sub["bce"], marker='o', label=split)
plt.xticks(rotation=30, ha='right')
plt.ylabel("validation BCE")
plt.title("Mean-response baselines for Task 2")
plt.legend()
savefig("baseline_bce_comparison.png")
plt.show()


## 9. Frequency smoothness and pattern-frequency structure

This checks whether frequency should be treated as a continuous variable rather than just a category.

In [ ]:
# Compute exact condition mean response vectors.
cond_rows = []
cond_vectors = []
for (p, f), grp in pd.DataFrame({"pattern": patterns, "frequency": freqs, "idx": np.arange(M)}).groupby(["pattern", "frequency"]):
    ids = grp["idx"].values
    mean_vec = X[ids].mean(axis=0).reshape(-1)
    cond_rows.append({"pattern": int(p), "frequency": float(f), "n": int(len(ids)), "vec_idx": len(cond_vectors)})
    cond_vectors.append(mean_vec)
cond_df = pd.DataFrame(cond_rows)
cond_mat = np.vstack(cond_vectors) if len(cond_vectors) else np.zeros((0, N*T))

freq_smooth_rows = []
for p, sub in cond_df.groupby("pattern"):
    sub = sub.sort_values("frequency")
    if len(sub) < 3:
        continue
    vecs = cond_mat[sub["vec_idx"].values]
    fs = sub["frequency"].values
    for i in range(len(sub)-1):
        corr = safe_corr(vecs[i], vecs[i+1])
        freq_smooth_rows.append({
            "pattern": int(p),
            "f1": float(fs[i]),
            "f2": float(fs[i+1]),
            "df": float(fs[i+1]-fs[i]),
            "adjacent_condition_mean_corr": corr,
        })
freq_smooth_df = pd.DataFrame(freq_smooth_rows)
if len(freq_smooth_df):
    display(freq_smooth_df.describe())
    summary["frequency_smoothness"] = {
        "mean_adjacent_freq_condition_corr": float(freq_smooth_df["adjacent_condition_mean_corr"].mean()),
        "median_adjacent_freq_condition_corr": float(freq_smooth_df["adjacent_condition_mean_corr"].median()),
        "min_adjacent_freq_condition_corr": float(freq_smooth_df["adjacent_condition_mean_corr"].min()),
    }
    plt.figure(figsize=(7,4))
    plt.scatter(freq_smooth_df["df"], freq_smooth_df["adjacent_condition_mean_corr"], alpha=0.7)
    plt.xlabel("frequency gap between adjacent tested frequencies")
    plt.ylabel("corr(condition mean responses)")
    plt.title("Frequency smoothness within same stimulation pattern")
    savefig("frequency_smoothness.png")
    plt.show()
    print(json.dumps(summary["frequency_smoothness"], indent=2))
else:
    print("Not enough frequencies per pattern for frequency smoothness diagnostic.")


## 10. Trial history / state-dependence diagnostic

This checks whether previous-trial activity predicts current response beyond z. If it does, a history-aware model may be worth adding.

In [ ]:
# Sort by stimulation time if available; otherwise keep data order.
try:
    stim_order = np.argsort(np.asarray(stimulation_times).reshape(-1))
except Exception:
    stim_order = np.arange(M)

Xo = X[stim_order]
po = patterns[stim_order]
fo = freqs[stim_order]
current_total = Xo.sum(axis=(1,2))
prev_total = np.r_[np.nan, current_total[:-1]]
prev_same_pattern = np.r_[False, po[1:] == po[:-1]]
try:
    stim_times_o = np.asarray(stimulation_times).reshape(-1)[stim_order]
    delta_t = np.r_[np.nan, np.diff(stim_times_o)]
except Exception:
    delta_t = np.full(M, np.nan)

valid = np.isfinite(prev_total)
summary["history"] = {
    "corr_prev_total_vs_current_total": safe_corr(prev_total[valid], current_total[valid]),
    "mean_current_total_after_prev_top_quartile": float(np.nanmean(current_total[valid][prev_total[valid] >= np.nanquantile(prev_total[valid], 0.75)])),
    "mean_current_total_after_prev_bottom_quartile": float(np.nanmean(current_total[valid][prev_total[valid] <= np.nanquantile(prev_total[valid], 0.25)])),
}

# Residualize current total by exact condition mean, then test history correlation on residual.
cond = condition_keys(po, fo)
cond_mean_total = {}
for c in np.unique(cond):
    cond_mean_total[c] = float(current_total[cond == c].mean())
resid_total = current_total - np.array([cond_mean_total[c] for c in cond])
summary["history"]["corr_prev_total_vs_condition_residual_current_total"] = safe_corr(prev_total[valid], resid_total[valid])

plt.figure(figsize=(6,5))
plt.scatter(prev_total[valid], current_total[valid], s=10, alpha=0.35)
plt.xlabel("previous trial total spikes")
plt.ylabel("current trial total spikes")
plt.title("Trial-to-trial state dependence")
savefig("history_prev_vs_current_total.png")
plt.show()

plt.figure(figsize=(8,3))
plt.plot(current_total[:min(1000, M)], lw=0.8)
plt.xlabel("trial order")
plt.ylabel("total spikes")
plt.title("Total response over experiment order")
savefig("trial_total_over_order.png")
plt.show()

print(json.dumps(summary["history"], indent=2))


## 11. Low-rank structure of condition means and individual responses

This tests whether a latent decoder/low-rank output head is a good fit.

In [ ]:
# SVD on condition means: conditions x (N*T)
if cond_mat.shape[0] >= 3:
    Y = cond_mat - cond_mat.mean(axis=0, keepdims=True)
    # economy SVD; cond count should be modest
    U, S, Vt = np.linalg.svd(Y, full_matrices=False)
    explained = (S**2) / np.sum(S**2) if np.sum(S**2) > 0 else np.zeros_like(S)
    cum = np.cumsum(explained)
    for k in [1,2,4,8,16,32,64]:
        if k <= len(cum):
            summary[f"condition_mean_svd_cumvar_{k}"] = float(cum[k-1])
    plt.figure(figsize=(7,4))
    plt.plot(np.arange(1, min(50, len(cum))+1), cum[:min(50, len(cum))], marker='o')
    plt.xlabel("number of components")
    plt.ylabel("cumulative variance of condition means")
    plt.title("Low-rank structure across condition mean responses")
    savefig("condition_mean_svd_cumvar.png")
    plt.show()

# SVD on sampled individual responses can be large; sample if needed.
sample_trials = np.arange(M)
if N_RANDOM_TRIALS is not None and M > N_RANDOM_TRIALS:
    sample_trials = rng.choice(M, size=N_RANDOM_TRIALS, replace=False)
Ytrial = X[sample_trials].reshape(len(sample_trials), -1)
Ytrial = Ytrial - Ytrial.mean(axis=0, keepdims=True)
try:
    U2, S2, Vt2 = np.linalg.svd(Ytrial, full_matrices=False)
    explained2 = (S2**2) / np.sum(S2**2) if np.sum(S2**2) > 0 else np.zeros_like(S2)
    cum2 = np.cumsum(explained2)
    for k in [1,2,4,8,16,32,64,128]:
        if k <= len(cum2):
            summary[f"trial_response_svd_cumvar_{k}"] = float(cum2[k-1])
    plt.figure(figsize=(7,4))
    plt.plot(np.arange(1, min(100, len(cum2))+1), cum2[:min(100, len(cum2))], marker='o')
    plt.xlabel("number of components")
    plt.ylabel("cumulative variance of individual responses")
    plt.title("Low-rank structure across individual trial responses")
    savefig("trial_response_svd_cumvar.png")
    plt.show()
except MemoryError:
    print("Skipping trial-level SVD due to memory pressure.")


## 12. Calibration and rare-spike warning

This checks whether a very low BCE could come mostly from predicting zeros. It also produces per-time and per-electrode BCE for the global baseline.

In [ ]:
global_p = X[train_idx].mean(axis=0)
valX = X[val_idx]
# Per-time and per-electrode BCE for global baseline.
p = clip_prob(global_p)
per_pos_loss = -(valX * np.log(p[None,:,:]) + (1-valX)*np.log(1-p[None,:,:]))
per_time_bce = per_pos_loss.mean(axis=(0,1))
per_elec_bce = per_pos_loss.mean(axis=(0,2))

summary["global_baseline_per_time_bce_peak_bin"] = int(np.argmax(per_time_bce))
summary["global_baseline_per_time_bce_first10_mean"] = float(per_time_bce[:min(10,T)].mean())
summary["global_baseline_per_time_bce_after20_mean"] = float(per_time_bce[min(20,T-1):].mean())

plt.figure(figsize=(9,3))
plt.plot(per_time_bce)
plt.xlabel("time bin")
plt.ylabel("BCE")
plt.title("Global-mean baseline BCE by time bin")
savefig("global_baseline_bce_by_time.png")
plt.show()

plt.figure(figsize=(8,3))
plt.hist(per_elec_bce, bins=40)
plt.xlabel("BCE per electrode")
plt.ylabel("electrode count")
plt.title("Global-mean baseline BCE by electrode")
savefig("global_baseline_bce_by_electrode.png")
plt.show()


## 13. Optional quick neural sanity check

Disabled by default. This is intentionally simple: it tests whether a residual MLP from condition to full output can beat the mean baselines on your split. It is not the final architecture.

In [ ]:
if not RUN_OPTIONAL_MODEL_SANITY_CHECKS:
    print("Skipping optional model sanity checks. Set RUN_OPTIONAL_MODEL_SANITY_CHECKS=True in the config cell to run them.")
else:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    from tqdm import tqdm

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    class ConditionMLP(nn.Module):
        def __init__(self, n_patterns, n_freq_features, n_out, hidden=256, emb=16):
            super().__init__()
            self.emb = nn.Embedding(n_patterns, emb)
            self.net = nn.Sequential(
                nn.Linear(emb + n_freq_features, hidden), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(hidden, n_out),
            )
        def forward(self, p, ff):
            return self.net(torch.cat([self.emb(p), ff], dim=-1))

    def make_freq_features(f_train, f_all):
        lo, hi = np.min(f_train), np.max(f_train)
        x = (f_all - lo) / max(hi - lo, 1e-6)
        lx = np.log1p(f_all)
        lx = (lx - np.log1p(f_train).mean()) / (np.log1p(f_train).std() + 1e-6)
        feats = [x, lx]
        for k in [1,2,4,8]:
            feats.append(np.sin(2*np.pi*k*x))
            feats.append(np.cos(2*np.pi*k*x))
        return np.stack(feats, axis=1).astype(np.float32)

    n_patterns = int(max(patterns.max()+1, 16))
    train_ff = make_freq_features(freqs[train_idx], freqs[train_idx])
    val_ff = make_freq_features(freqs[train_idx], freqs[val_idx])

    ytr = X[train_idx].reshape(len(train_idx), -1).astype(np.float32)
    yva = X[val_idx].reshape(len(val_idx), -1).astype(np.float32)

    baseline_bias = logit(X[train_idx].mean(axis=0)).reshape(-1).astype(np.float32)

    model = ConditionMLP(n_patterns, train_ff.shape[1], ytr.shape[1]).to(device)
    # initialize final layer near zero, then add baseline bias outside model
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    bias = torch.tensor(baseline_bias, device=device)

    ds = TensorDataset(torch.tensor(patterns[train_idx], dtype=torch.long), torch.tensor(train_ff), torch.tensor(ytr))
    dl = DataLoader(ds, batch_size=256, shuffle=True)
    val_p = torch.tensor(patterns[val_idx], dtype=torch.long, device=device)
    val_ff_t = torch.tensor(val_ff, device=device)
    val_y = torch.tensor(yva, device=device)
    best = np.inf
    for epoch in range(20):
        model.train()
        for bp, bf, by in dl:
            bp,bf,by = bp.to(device), bf.to(device), by.to(device)
            logits = bias[None,:] + model(bp,bf)
            loss = F.binary_cross_entropy_with_logits(logits, by)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_logits = bias[None,:] + model(val_p, val_ff_t)
            val_loss = F.binary_cross_entropy_with_logits(val_logits, val_y).item()
        best = min(best, val_loss)
        print(epoch, val_loss)
    summary["optional_condition_mlp_best_val_bce"] = float(best)


## 14. Final summary to send back

Run this cell and send me:

1. the printed JSON summary,
2. the `task2_assumption_audit_outputs` folder or at least the most important plots,
3. any weird warnings/errors that appeared above.

In [ ]:
# Add automatic interpretation flags. These are intentionally conservative heuristics.
flags = {}

if summary.get("coordinates_extracted"):
    eo = summary.get("electrode_order", {})
    flags["electrode_index_order_is_spatially_local"] = bool(
        eo.get("ratio_observed_to_random", 1.0) < 0.65 and
        eo.get("fraction_physical_NN_with_index_gap_le_3", 0.0) > 0.25
    )
    sc = summary.get("spatial_correlation", {})
    flags["physical_neighbors_have_stronger_response_correlation"] = bool(
        sc.get("nearest_minus_farthest_full_corr", 0.0) > 0.02 and
        sc.get("spearman_distance_vs_full_count_corr", 0.0) < -0.10
    )
else:
    flags["electrode_index_order_is_spatially_local"] = None
    flags["physical_neighbors_have_stronger_response_correlation"] = None

em = summary.get("early_mass", {})
first8 = em.get("first_8_bins_fraction_of_spikes", np.nan)
flags["response_is_strongly_early"] = bool(np.isfinite(first8) and first8 > 0.50)
flags["time_bins_are_correlated"] = bool(summary.get("temporal_adjacent_corr_mean", 0.0) > 0.05)

# Baseline flags.
bdf = pd.DataFrame(summary.get("baseline_bce", []))
if len(bdf):
    rnd = bdf[bdf["split"] == "random_trial"].copy()
    if len(rnd):
        g = float(rnd[rnd["baseline"] == "global"]["bce"].iloc[0])
        best = float(rnd["bce"].min())
        flags["condition_means_beat_global_mean_on_random_split"] = bool((g - best) / g > 0.03)
        flags["condition_signal_random_split_relative_bce_improvement"] = float((g - best) / g)
    lf = bdf[bdf["split"] == "leave_frequency_out"].copy()
    if len(lf):
        g = float(lf[lf["baseline"] == "global"]["bce"].iloc[0])
        best = float(lf["bce"].min())
        flags["condition_signal_leave_frequency_out_relative_bce_improvement"] = float((g - best) / g)

hist = summary.get("history", {})
flags["history_state_may_matter"] = bool(abs(hist.get("corr_prev_total_vs_condition_residual_current_total", 0.0)) > 0.08)

fs = summary.get("frequency_smoothness", {})
flags["frequency_effect_looks_smooth"] = bool(fs.get("median_adjacent_freq_condition_corr", 0.0) > 0.30)

summary["interpretation_flags"] = flags

summary_path = os.path.join(OUTPUT_DIR, "task2_assumption_audit_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("FINAL SUMMARY")
print(json.dumps(summary, indent=2))
print("\nSaved summary to:", summary_path)
print("Saved plots to:", OUTPUT_DIR)
